In [1]:
import tensorflow as tf

In [2]:
from keras import layers

### Sequence-to-sequence learning

#### English-to-Spanish translation

In [3]:
### Dataset from manythings.org
!wget http://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip
!unzip -q fra-eng.zip

--2026-05-13 01:22:49--  http://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 142.251.2.207, 74.125.137.207, 142.250.101.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.251.2.207|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3423204 (3.3M) [application/zip]
Saving to: ‘fra-eng.zip’

fra-eng.zip         100%[===================>]   3.26M  --.-KB/s    in 0.06s   

2026-05-13 01:22:49 (58.1 MB/s) - ‘fra-eng.zip’ saved [3423204/3423204]



In [4]:
text_file = "fra.txt"
with open(text_file) as f:
    lines = f.read().split("\n")[:-1]
text_pairs = []
for line in lines:
    english, french = line.split("\t")
    french = "[start] " + french + " [end]"
    text_pairs.append((english, french))

In [5]:
import random
random.choice(text_pairs)

("Have you been told the reasons why we didn't hire you?",
 "[start] T'a-t-on dit les raisons pour lesquelles nous ne t'avons pas embauché ? [end]")

In [6]:
import random

random.shuffle(text_pairs)
val_samples = int(0.15 * len(text_pairs))
train_samples = len(text_pairs) - 2 * val_samples
train_pairs = text_pairs[:train_samples]
val_pairs = text_pairs[train_samples : train_samples + val_samples]
test_pairs = text_pairs[train_samples + val_samples :]

### String tokenization

In [8]:
import string
import re

strip_chars = string.punctuation
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", ""
    )

vocab_size = 20000
sequence_length = 30

english_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)
french_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)
train_english_texts = [pair[0] for pair in train_pairs]
train_french_texts = [pair[1] for pair in train_pairs]
english_tokenizer.adapt(train_english_texts)
french_tokenizer.adapt(train_french_texts)

In [9]:
batch_size = 64

def format_dataset(eng, french):
    eng = english_tokenizer(eng)
    french = french_tokenizer(french)
    features = {"english": eng, "french": french[:, :-1]}
    labels = french[:, 1:]
    sample_weights = labels != 0
    return features, labels, sample_weights

def make_dataset(pairs):
    eng_texts, french_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    french_texts = list(french_texts)
    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, french_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset, num_parallel_calls=4)
    return dataset.shuffle(2048).cache()

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

In [10]:
inputs, targets, sample_weights = next(iter(train_ds))
print(inputs["english"].shape)

(64, 30)


In [11]:
print(inputs["french"].shape)

(64, 30)


In [12]:
print(targets.shape)

(64, 30)


In [13]:
print(sample_weights.shape)

(64, 30)


#### Sequence-to-sequence learning with RNNs

In [15]:
import keras

In [16]:
embed_dim = 256
hidden_dim = 1024

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(source)
rnn_layer = layers.GRU(hidden_dim)
rnn_layer = layers.Bidirectional(rnn_layer, merge_mode="sum")
encoder_output = rnn_layer(x)

In [17]:
target = keras.Input(shape=(None,), dtype="int32", name="french")
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(target)
rnn_layer = layers.GRU(hidden_dim, return_sequences=True)
x = rnn_layer(x, initial_state=encoder_output)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
seq2seq_rnn = keras.Model([source, target], target_predictions)

In [18]:
seq2seq_rnn.summary(line_length=80)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ english (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ french (InputLayer)   │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding (Embedding) │ (None, None, 256) │   5,120,000 │ english[0][0]      │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal (NotEqual)  │ (None, None)      │           0 │ english[0][0]      │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_1           │ (None, None, 256) │   5,120,000 │ french[0][0]       │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ bidirectional         │ (None, 1024)      │   7,876,608 │ embedding[0][0],   │
│ (Bidirectional)       │                   │             │ not_equal[0][0]    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ gru_1 (GRU)           │ (None, None,      │   3,938,304 │ embedding_1[0][0], │
│                       │ 1024)             │             │ bidirectional[0][… │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dropout (Dropout)     │ (None, None,      │           0 │ gru_1[0][0]        │
│                       │ 1024)             │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dense (Dense)         │ (None, None,      │  20,500,000 │ dropout[0][0]      │
│                       │ 20000)            │             │                    │
└───────────────────────┴───────────────────┴─────────────┴────────────────────┘

 Total params: 42,554,912 (162.33 MB)

 Trainable params: 42,554,912 (162.33 MB)

 Non-trainable params: 0 (0.00 B)

In [21]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
    "seq2seq_french_eng_translation_model.keras", save_best_only=True
),
    keras.callbacks.EarlyStopping(patience=5, monitor="val_loss"),
]

In [23]:

seq2seq_rnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
seq2seq_rnn.fit(
    train_ds,
    epochs=15,
    validation_data=val_ds,
    callbacks=callbacks
  )

Epoch 1/15
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 135s 71ms/step - accuracy: 0.3721 - loss: 3.5350 - val_accuracy: 0.5095 - val_loss: 2.3489
Epoch 2/15
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 129s 71ms/step - accuracy: 0.5472 - loss: 2.1242 - val_accuracy: 0.6068 - val_loss: 1.7486
Epoch 3/15
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 129s 71ms/step - accuracy: 0.6292 - loss: 1.5471 - val_accuracy: 0.6456 - val_loss: 1.5325
Epoch 4/15
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 129s 70ms/step - accuracy: 0.6804 - loss: 1.2141 - val_accuracy: 0.6630 - val_loss: 1.4569
Epoch 5/15
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 129s 71ms/step - accuracy: 0.7185 - loss: 0.9983 - val_accuracy: 0.6741 - val_loss: 1.4328
Epoch 6/15
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 127s 70ms/step - accuracy: 0.7481 - loss: 0.8569 - val_accuracy: 0.6794 - val_loss: 1.4414
Epoch 7/15
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 127s 70ms/step - accuracy: 0.7690 - loss: 0.7628 - val_accuracy: 0.6833 - val_loss: 1.4484
Epoch 8/15
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 127s 70ms/step - accuracy: 

In [24]:
seq2seq_rnn.load_weights("seq2seq_french_eng_translation_model.keras")

In [25]:
import numpy as np

french_vocab = french_tokenizer.get_vocabulary()
french_index_lookup = dict(zip(range(len(french_vocab)), french_vocab))

def generate_translation(input_sentence):
    tokenized_input_sentence = english_tokenizer([input_sentence])
    decoded_sentence = "[start]"
    for i in range(sequence_length):
        tokenized_target_sentence = french_tokenizer([decoded_sentence])
        inputs = [tokenized_input_sentence, tokenized_target_sentence]
        next_token_predictions = seq2seq_rnn.predict(inputs, verbose=0)
        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = french_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(5):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(generate_translation(input_sentence))

-
My father may be sleeping.
[start] mon père peut être dormir [end]
-
This building belongs to my family.
[start] ce bâtiment appartient à ma famille [end]
-
Did you see anybody there?
[start] avezvous vu qui que ce soit [end]
-
They were swimming.
[start] ils nageaient [end]
-
Take your umbrella with you.
[start] prends ton parapluie avec toi [end]


### The Transformer architecture

#### Dot-product attention

#### Transformer encoder block

In [55]:
class TransformerEncoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()

    def call(self, source, source_mask):
        residual = x = source
        mask = source_mask[:, None, :]
        x = self.self_attention(query=x, key=x, value=x, attention_mask=mask)
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

#### Transformer decoder block

In [56]:
class TransformerDecoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()
        self.cross_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.cross_attention_layernorm = layers.LayerNormalization()
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()

    def call(self, target, source, source_mask):
        residual = x = target
        x = self.self_attention(query=x, key=x, value=x, use_causal_mask=True)
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x
        mask = source_mask[:, None, :]
        x = self.cross_attention(
            query=x, key=source, value=source, attention_mask=mask
        )
        x = x + residual
        x = self.cross_attention_layernorm(x)
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

#### Sequence-to-sequence learning with a Transformer

In [57]:
hidden_dim = 256
intermediate_dim = 2048
num_heads = 8

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = layers.Embedding(vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(
    source=x,
    source_mask=source != 0,
)

target = keras.Input(shape=(None,), dtype="int32", name="french")
x = layers.Embedding(vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(
    target=x,
    source=encoder_output,
    source_mask=source != 0,
)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([source, target], target_predictions)

In [58]:
transformer.summary(line_length=80)

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ english (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_18          │ (None, None, 256) │   5,120,000 │ english[0][0]      │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal_12          │ (None, None)      │           0 │ english[0][0]      │
│ (NotEqual)            │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ french (InputLayer)   │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_encoder_5 │ (None, None, 256) │   1,315,072 │ embedding_18[0][0… │
│ (TransformerEncoder)  │                   │             │ not_equal_12[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal_13          │ (None, None)      │           0 │ english[0][0]      │
│ (NotEqual)            │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_19          │ (None, None, 256) │   5,120,000 │ french[0][0]       │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_decoder_5 │ (None, None, 256) │   1,578,752 │ transformer_encod… │
│ (TransformerDecoder)  │                   │             │ not_equal_13[0][0… │
│                       │                   │             │ embedding_19[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dropout_24 (Dropout)  │ (None, None, 256) │           0 │ transformer_decod… │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dense_30 (Dense)      │ (None, None,      │   5,140,000 │ dropout_24[0][0]   │
│                       │ 20000)            │             │                    │
└───────────────────────┴───────────────────┴─────────────┴────────────────────┘

 Total params: 18,273,824 (69.71 MB)

 Trainable params: 18,273,824 (69.71 MB)

 Non-trainable params: 0 (0.00 B)

In [59]:
transformer_callbacks = [
    keras.callbacks.ModelCheckpoint(
    "transformer_french_eng_translation_model.keras", save_best_only=True
),
    keras.callbacks.EarlyStopping(patience=10, monitor="val_loss"),
]

#### Embedding positional information

In [60]:
from keras import ops

class PositionalEmbedding(keras.Layer):
    def __init__(self, sequence_length, input_dim, output_dim):
        super().__init__()
        self.token_embeddings = layers.Embedding(input_dim, output_dim)
        self.position_embeddings = layers.Embedding(sequence_length, output_dim)

    def call(self, inputs):
        positions = ops.cumsum(ops.ones_like(inputs), axis=-1) - 1
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

In [61]:
hidden_dim = 256
intermediate_dim = 2056
num_heads = 8

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(
    source=x,
    source_mask=source != 0,
)

target = keras.Input(shape=(None,), dtype="int32", name="french")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(
    target=x,
    source=encoder_output,
    source_mask=source != 0,
)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([source, target], target_predictions)

In [62]:
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)


In [63]:
transformer.fit(
    train_ds,
    epochs=30,
    validation_data=val_ds,
    callbacks=transformer_callbacks
    )

Epoch 1/30
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 42s 14ms/step - accuracy: 0.4242 - loss: 0.9151 - val_accuracy: 0.5679 - val_loss: 0.5956
Epoch 2/30
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - accuracy: 0.5975 - loss: 0.5630 - val_accuracy: 0.6369 - val_loss: 0.4737
Epoch 3/30
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - accuracy: 0.6577 - loss: 0.4433 - val_accuracy: 0.6663 - val_loss: 0.4225
Epoch 4/30
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - accuracy: 0.6927 - loss: 0.3742 - val_accuracy: 0.6804 - val_loss: 0.4013
Epoch 5/30
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - accuracy: 0.7185 - loss: 0.3274 - val_accuracy: 0.6899 - val_loss: 0.3922
Epoch 6/30
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - accuracy: 0.7376 - loss: 0.2928 - val_accuracy: 0.6965 - val_loss: 0.3839
Epoch 7/30
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - accuracy: 0.7543 - loss: 0.2656 - val_accuracy: 0.7022 - val_loss: 0.3826
Epoch 8/30
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - accuracy: 0.7689 -

In [42]:
transformer.load_weights("transformer_french_eng_translation_model.keras")

### Generate predictions

In [64]:
import numpy as np

spa_vocab = french_tokenizer.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))

def generate_translation(input_sentence):
    tokenized_input_sentence = english_tokenizer([input_sentence])
    decoded_sentence = "[start]"
    for i in range(sequence_length):
        tokenized_target_sentence = french_tokenizer([decoded_sentence])
        tokenized_target_sentence = tokenized_target_sentence[:, :-1]
        inputs = [tokenized_input_sentence, tokenized_target_sentence]
        next_token_predictions = transformer.predict(inputs, verbose=0)
        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(15):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(generate_translation(input_sentence))

-
It is the job that is never started that takes longest to finish.
[start] cest le travail qui ne sest jamais mis à [UNK] aussi long de finir [end]
-
I don't want to be told what to do.
[start] je ne veux pas être dit quoi faire [end]
-
He is always reading.
[start] il est toujours en train de lire [end]
-
Your things are all here.
[start] tes affaires sont toutes ici [end]
-
I think it's time for me to abandon that plan.
[start] je pense quil est temps pour moi dabandonner ce projet [end]
-
Are you busy?
[start] estu occupé [end]
-
You should take an umbrella with you this morning.
[start] tu devrais prendre un parapluie avec toi ce matin [end]
-
Aren't you going to open the box?
[start] nallezvous pas de fermer la boîte [end]
-
This plan is far from perfect, but it's the best plan we have.
[start] ce projet est loin dêtre parfait mais elle est en mesure que nous ayons le meilleur plan [end]
-
I can understand him perfectly.
[start] je peux le comprendre parfaitement [end]
-
The bear

---
## Save EN→FR Models & Vocabularies for the Web App

The web app expects models named `rnn_en_fr.keras` / `transformer_en_fr.keras`
and four vocabulary JSON files.  Run this section after the EN→FR training above.


In [ ]:
import json, shutil

# ── Save EN→FR vocabularies ───────────────────────────────────────────────
with open('vocab_en_src.json', 'w') as f:
    json.dump(english_tokenizer.get_vocabulary(), f, ensure_ascii=False)

with open('vocab_fr_tgt.json', 'w') as f:
    json.dump(french_tokenizer.get_vocabulary(), f, ensure_ascii=False)

# ── Save shared config ────────────────────────────────────────────────────
config = {'vocab_size': vocab_size, 'seq_len': sequence_length, 'batch_size': batch_size}
with open('config.json', 'w') as f:
    json.dump(config, f, indent=2)

# ── Copy EN→FR model files with web-app names ─────────────────────────────
shutil.copy('seq2seq_french_eng_translation_model.keras', 'rnn_en_fr.keras')
shutil.copy('transformer_french_eng_translation_model.keras', 'transformer_en_fr.keras')

print('EN→FR artefacts ready:')
print('  vocab_en_src.json, vocab_fr_tgt.json, config.json')
print('  rnn_en_fr.keras, transformer_en_fr.keras')


---
## French → English Translation

We reuse the same train/val/test splits but swap source and target languages.
Two new tokenizers are built:
- **`fr_source_tokenizer`** — adapted on plain French text (source side)
- **`en_target_tokenizer`** — adapted on `[start] english [end]` text (target side)


In [ ]:
# Reuse the same pairs and splits from above.
# FR→EN: source = French (plain), target = [start] English [end]

fr_source_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=sequence_length,
    standardize=custom_standardization,
)
en_target_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,  # preserves [ and ] for sentinels
)

train_fr_source_texts = [pair[1] for pair in train_pairs]   # plain French
train_en_target_texts = ['[start] ' + pair[0] + ' [end]' for pair in train_pairs]

fr_source_tokenizer.adapt(train_fr_source_texts)
en_target_tokenizer.adapt(train_en_target_texts)

print('FR source vocab size:', len(fr_source_tokenizer.get_vocabulary()))
print('EN target vocab size:', len(en_target_tokenizer.get_vocabulary()))


In [ ]:
# ── Save FR→EN vocabularies ───────────────────────────────────────────────
with open('vocab_fr_src.json', 'w') as f:
    json.dump(fr_source_tokenizer.get_vocabulary(), f, ensure_ascii=False)

with open('vocab_en_tgt.json', 'w') as f:
    json.dump(en_target_tokenizer.get_vocabulary(), f, ensure_ascii=False)

print('Saved vocab_fr_src.json and vocab_en_tgt.json')


In [ ]:
# ── Build FR→EN tf.data datasets ─────────────────────────────────────────

def format_fr_en_dataset(fr, en):
    fr_enc = fr_source_tokenizer(fr)
    en_enc = en_target_tokenizer(en)
    features = {'source': fr_enc, 'target': en_enc[:, :-1]}
    labels   = en_enc[:, 1:]
    weights  = labels != 0
    return features, labels, weights

def make_fr_en_dataset(pairs):
    fr_texts = [p[1] for p in pairs]
    en_texts = ['[start] ' + p[0] + ' [end]' for p in pairs]
    ds = tf.data.Dataset.from_tensor_slices((fr_texts, en_texts))
    ds = ds.batch(batch_size)
    ds = ds.map(format_fr_en_dataset, num_parallel_calls=4)
    return ds.shuffle(2048).cache()

fr_en_train_ds = make_fr_en_dataset(train_pairs)
fr_en_val_ds   = make_fr_en_dataset(val_pairs)

# Sanity check
feats, labs, wts = next(iter(fr_en_train_ds))
print('source shape:', feats['source'].shape)   # (64, 30)
print('target shape:', feats['target'].shape)   # (64, 30)
print('labels shape:', labs.shape)              # (64, 30)


### FR→EN — GRU Seq2Seq

Same bidirectional-GRU architecture, inputs now named `source` / `target`
to match the web-app convention.


In [ ]:
embed_dim  = 256
hidden_dim = 1024

source = keras.Input(shape=(None,), dtype='int32', name='source')
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(source)
encoder_output = layers.Bidirectional(layers.GRU(hidden_dim), merge_mode='sum')(x)

target = keras.Input(shape=(None,), dtype='int32', name='target')
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(target)
x = layers.GRU(hidden_dim, return_sequences=True)(x, initial_state=encoder_output)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation='softmax')(x)

rnn_fr_en = keras.Model([source, target], target_predictions)
rnn_fr_en.summary(line_length=80)


In [ ]:
rnn_fr_en_callbacks = [
    keras.callbacks.ModelCheckpoint(
        'rnn_fr_en.keras', save_best_only=True
    ),
    keras.callbacks.EarlyStopping(patience=5, monitor='val_loss', restore_best_weights=True),
]

rnn_fr_en.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    weighted_metrics=['accuracy'],
)
rnn_fr_en.fit(
    fr_en_train_ds,
    epochs=15,
    validation_data=fr_en_val_ds,
    callbacks=rnn_fr_en_callbacks,
)


In [ ]:
rnn_fr_en.load_weights('rnn_fr_en.keras')


In [ ]:
# ── FR→EN RNN evaluation ─────────────────────────────────────────────────
en_vocab     = en_target_tokenizer.get_vocabulary()
en_idx_lookup = dict(zip(range(len(en_vocab)), en_vocab))

def rnn_translate_fr_en(input_sentence):
    tokenized_input = fr_source_tokenizer([input_sentence])
    decoded = '[start]'
    for i in range(sequence_length):
        tokenized_target = en_target_tokenizer([decoded])
        preds = rnn_fr_en.predict(
            {'source': tokenized_input, 'target': tokenized_target}, verbose=0
        )
        next_idx   = np.argmax(preds[0, i, :])
        next_token = en_idx_lookup[next_idx]
        decoded   += ' ' + next_token
        if next_token == '[end]':
            break
    tokens = [t for t in decoded.split() if t not in ('[start]', '[end]')]
    return ' '.join(tokens)

test_fr_texts = [pair[1] for pair in test_pairs]
print('=== FR→EN  GRU RNN ===')
for _ in range(5):
    src = random.choice(test_fr_texts)
    print('-')
    print('FR:', src)
    print('EN:', rnn_translate_fr_en(src))


### FR→EN — Transformer

Reuses the `TransformerEncoder`, `TransformerDecoder`, and `PositionalEmbedding`
classes defined earlier in this notebook.


In [ ]:
hidden_dim       = 256
intermediate_dim = 2056
num_heads        = 8

source = keras.Input(shape=(None,), dtype='int32', name='source')
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(
    source=x, source_mask=source != 0
)

target = keras.Input(shape=(None,), dtype='int32', name='target')
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(
    target=x, source=encoder_output, source_mask=source != 0
)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation='softmax')(x)

transformer_fr_en = keras.Model([source, target], target_predictions)
transformer_fr_en.summary(line_length=80)


In [ ]:
transformer_fr_en_callbacks = [
    keras.callbacks.ModelCheckpoint(
        'transformer_fr_en.keras', save_best_only=True
    ),
    keras.callbacks.EarlyStopping(patience=5, monitor='val_loss', restore_best_weights=True),
]

transformer_fr_en.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    weighted_metrics=['accuracy'],
)
transformer_fr_en.fit(
    fr_en_train_ds,
    epochs=30,
    validation_data=fr_en_val_ds,
    callbacks=transformer_fr_en_callbacks,
)


In [ ]:
transformer_fr_en.load_weights('transformer_fr_en.keras')


In [ ]:
# ── FR→EN Transformer evaluation ─────────────────────────────────────────
def transformer_translate_fr_en(input_sentence):
    tokenized_input = fr_source_tokenizer([input_sentence])
    decoded = '[start]'
    for i in range(sequence_length):
        tokenized_target = en_target_tokenizer([decoded])[:, :-1]
        preds = transformer_fr_en.predict(
            {'source': tokenized_input, 'target': tokenized_target}, verbose=0
        )
        next_idx   = np.argmax(preds[0, i, :])
        next_token = en_idx_lookup[next_idx]
        decoded   += ' ' + next_token
        if next_token == '[end]':
            break
    tokens = [t for t in decoded.split() if t not in ('[start]', '[end]')]
    return ' '.join(tokens)

print('=== FR→EN  Transformer ===')
for _ in range(5):
    src = random.choice(test_fr_texts)
    print('-')
    print('FR:', src)
    print('EN:', transformer_translate_fr_en(src))


---
## Export All Artefacts to Google Drive

Colab VMs are ephemeral — copy everything to Drive so files persist
after the session ends.  Then download them to your local `models/` folder
and run `uvicorn backend.app:app` to start the web app.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, shutil

save_dir = '/content/drive/MyDrive/translation_models'
os.makedirs(save_dir, exist_ok=True)

artefacts = [
    # Models
    'rnn_en_fr.keras',
    'transformer_en_fr.keras',
    'rnn_fr_en.keras',
    'transformer_fr_en.keras',
    # Vocabularies
    'vocab_en_src.json',
    'vocab_fr_tgt.json',
    'vocab_fr_src.json',
    'vocab_en_tgt.json',
    # Config
    'config.json',
]

for name in artefacts:
    if os.path.exists(name):
        dest = os.path.join(save_dir, name)
        shutil.copy(name, dest)
        size_mb = os.path.getsize(dest) / 1e6
        print(f'  ✓  {name}  ({size_mb:.1f} MB)')
    else:
        print(f'  ✗  {name}  — not found (skipped)')

print(f'\nAll artefacts saved to {save_dir}')
print('Download them to your local models/ directory and start the web app:')
print('  uvicorn backend.app:app --reload')
